In [1]:
from datasets import load_dataset

ds = load_dataset("AIGuruTinix/ViBioMIR", 'corpus')


/Users/buithianhdao/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 4394718/4394718 [00:01<00:00, 3915254.50 examples/s]


In [ ]:
! pip install playwright
! playwright install


In [ ]:
!playwright install-deps chromium


In [ ]:
!playwright install chromium


In [ ]:
! pip install --upgrade trafilatura


In [ ]:
import asyncio
import json
from urllib.parse import urlparse

import trafilatura
from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
    Error as PlaywrightError,
)


async def extract_url(url: str) -> dict:
    """
    Render URL bằng Playwright, sau đó dùng Trafilatura
    để extract main article + metadata.

    Có xử lý:
    - navigation chậm
    - redirect / page đang thay đổi DOM
    - JS client-side render
    - Trafilatura extract thất bại
    - nội dung article quá ngắn / rỗng
    """

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        page.set_default_timeout(15_000)

        try:
            # 1. Bắt đầu navigation.
            # Chỉ chờ đến commit để tránh bị kẹt bởi resource bên thứ ba.
            try:
                await page.goto(
                    url,
                    wait_until="commit",
                    timeout=30_000,
                )
            except PlaywrightTimeoutError:
                # Có thể page đã load được một phần,
                # nên vẫn thử xử lý tiếp.
                pass

            # 2. Chờ document cơ bản tồn tại.
            try:
                await page.wait_for_selector(
                    "body",
                    state="attached",
                    timeout=10_000,
                )
            except PlaywrightTimeoutError:
                pass

            # 3. Cho JS / redirect / client-side rendering chạy một chút.
            await asyncio.sleep(2)

            # 4. Thử chờ DOM parse xong.
            # Không bắt buộc, để tránh bị treo.
            try:
                await page.wait_for_load_state(
                    "domcontentloaded",
                    timeout=10_000,
                )
            except PlaywrightTimeoutError:
                pass

            # 5. Lấy HTML.
            # Một số site có thể đang redirect/navigation đúng lúc này,
            # nên retry vài lần.
            html = None
            last_error = None

            for _ in range(3):
                try:
                    html = await page.content()
                    break

                except PlaywrightError as e:
                    last_error = e
                    await asyncio.sleep(1)

            if html is None:
                raise RuntimeError(
                    f"Unable to retrieve stable page content: {last_error}"
                )

        except Exception as e:
            return {
                "url": url,
                "domain": urlparse(url).netloc,
                "success": False,
                "error": str(e),
            }

        finally:
            await browser.close()

    # 6. Extract article bằng Trafilatura.
    result = trafilatura.extract(
        html,
        url=url,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_tables=True,
        favor_precision=True,
    )

    if result is None:
        return {
            "url": url,
            "domain": urlparse(url).netloc,
            "success": False,
            "error": "Trafilatura could not extract content",
        }

    try:
        data = json.loads(result)
    except json.JSONDecodeError as e:
        return {
            "url": url,
            "domain": urlparse(url).netloc,
            "success": False,
            "error": f"Invalid Trafilatura JSON: {e}",
        }

    # 7. Validate nội dung article.
    text = (data.get("text") or "").strip()

    if len(text) < 200:
        return {
            "url": data.get("url") or url,
            "domain": urlparse(url).netloc,
            "source": data.get("sitename") or data.get("hostname"),
            "title": data.get("title"),
            "text_length": len(text),
            "success": False,
            "error": "Extracted article content is empty or too short",
        }

    # 8. Output chuẩn hóa.
    return {
        "url": data.get("url") or url,
        "domain": urlparse(url).netloc,
        "source": data.get("sitename") or data.get("hostname"),
        "title": data.get("title"),
        "author": data.get("author"),
        "published_date": data.get("date"),
        "description": data.get("description"),
        "text": text,
        "success": True,
    }


In [ ]:
import pandas as pd
r2ai_lucraftmeds = pd.DataFrame(columns=["doc_id","url","domain","source","title","author","published_date","description","text","success"])


In [ ]:
r2ai_lucraftmeds


In [ ]:
urls = []
for i in range(len(ds["train"])):
    urls.append(ds["train"][i])


In [ ]:
len(urls)


In [ ]:
urls[0]


In [ ]:
from tqdm.auto import tqdm
import pandas as pd
rows = []
errors = []

for item in tqdm(urls, desc="Crawling"):
    doc_id = item["id"]
    url = item["url"]

    article = await extract_url(url)

    if "error" in article:
        errors.append(article)
        print("Errors found at: ",errors)
        continue
        
    article["doc_id"] = doc_id
    rows.append(article)

r2ai_lucraftmeds = pd.DataFrame(rows)
r2ai_sub_errors = pd.DataFrame(errors)


In [ ]:
len(rows)


In [ ]:
len(rows[0]['text'])


In [ ]:
r2ai_lucraftmeds.to_csv('R2AIS3-Raw_Medical_Corpus.csv',index=False)
r2ai_sub_errors.to_csv("R2AIS3-Errors.csv",index=False)
